# **Multi-Agent Reinforcement Learning**



---
## Nash Equilibrium (Theory)

A Nash Equilibrium (NE) represents a state where no player can improve their outcome by unilaterally changing their strategy. For our games, we'll focus on finding the mixed-strategy NE, where players choose their actions probabilistically.

### 1.1 Standard Medieval Warfare (Cavalry-Infantry-Archers)

In this tactical game, two generals face off. Cavalry beats Archers, Archers beat Infantry, and Infantry beats Cavalry. Given the standard payoff matrix:

| Player 1 | Cavalry | Infantry | Archers |
| :--- | :--: | :---: | :---: |
| **Cavalry** | 0, 0 | -1, 1 | 1, -1 |
| **Infantry**| 1, -1 | 0, 0 | -1, 1 |
| **Archers** | -1, 1 | 1, -1 | 0, 0 |

**Your Task:** Analytically derive the mixed-strategy Nash Equilibrium for this game. Show the steps for setting up the indifference equations for Player 1 and solving for Player 2's equilibrium strategy probabilities $(q_C, q_I, q_A)$. (Find the Mixed Nash equilibrium of the game).

### 1.2 Modified Medieval Warfare

Now, consider a modified game where the terrain alters the combat effectiveness. A Cavalry charge against Archers is absolutely devastating, but Archers have a stronger high-ground advantage against Infantry:

| Player 1 | Cavalry | Infantry | Archers |
| :--- | :--: | :---: | :---: |
| **Cavalry** | 0, 0 | -1, 1 | 3, -3 |
| **Infantry**| 1, -1 | 0, 0 | -2, 2 |
| **Archers** | -3, 3 | 2, -2 | 0, 0 |

**Your Task:** Like the previous one, derive the mixed-strategy Nash Equilibrium for this modified game. Show your work as you calculate the new probability distribution.

## Your Answer

### 1.1 Standard Medieval Warfare

Let Player 2 use $q=(q_C,q_I,q_A)$. Player 1's expected payoffs are:

$$
U_1(C)=-q_I+q_A,\quad
U_1(I)=q_C-q_A,\quad
U_1(A)=-q_C+q_I
$$

At equilibrium, all three payoffs are equal and $q_C+q_I+q_A=1$. Solving gives:

$$
q_C=q_I=q_A=\frac{1}{3}
$$

By symmetry, both players use the same strategy:

$$
p^*=q^*=\left(\frac13,\frac13,\frac13\right)
$$

The game value is $0$, so no unilateral pure deviation gives a higher payoff.

### 1.2 Modified Medieval Warfare

For the modified game:

$$
U_1(C)=-q_I+3q_A,\quad
U_1(I)=q_C-2q_A,\quad
U_1(A)=-3q_C+2q_I
$$

Setting the payoffs equal with $q_C+q_I+q_A=1$ gives:

$$
q_C=\frac13,\quad q_I=\frac12,\quad q_A=\frac16
$$

As a result:

$$
p^*=q^*=\left(\frac13,\frac12,\frac16\right)
$$

The game value is again $0$. The equilibrium probabilities change because the payoff magnitudes change, so different probabilities are needed to keep all three actions equally valuable in expectation.

## Problem 2: Learning by Observation - Fictitious Play (Implementation)

Fictitious Play is an intuitive learning algorithm where each agent models its opponent as playing a stationary strategy defined by the historical frequency of their past actions. The agent then plays a **best response** to this belief.

### 2.1 Implementation

**Your Task:** Implement the `simulate_fictitious_play` function below. It should take the payoff matrices for both players and the number of iterations as input. At each step, each player should choose the action that maximizes their expected payoff given the history of the opponent's plays.

**Algorithm:** At each time step $t > 0$, Player $i$ forms a belief that their opponent ($-i$) will play each action $a'$ with a probability equal to its historical frequency. The agent then chooses an action $a_i^*$ that is a best response to this belief.

Let $C_{t-1}(a_{-i})$ be the count of times opponent $-i$ has played action $a_{-i}$ up to step $t-1$. Player $i$'s best response is:
$$a_{i,t}^* = \arg\max_{a_i \in A_i} \sum_{a_{-i} \in A_{-i}} u_i(a_i, a_{-i}) \cdot \frac{C_{t-1}(a_{-i})}{t-1}$$

**Note on Tie-Breaking:** If multiple actions yield the same maximal expected payoff, your agent should choose one of these best responses uniformly at random.

**Your Task:**
1.  Run your simulation for **1,000,000 iterations** on both the **standard** and **modified** RSP games.
2.  Generate two plots, one for each game. Each plot should show the evolution of Players action frequencies over time and include horizontal lines indicating the theoretical NE probabilities you calculated in Problem 1.
3.  **Analyze your results:** Do the action frequencies converge? If so, do they converge to the Nash Equilibrium? Explain the observed behavior.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def simulate_fictitious_play(payoff_A, payoff_B, iterations):
    """
    Simulates the Fictitious Play algorithm for two players.

    Parameters:
    payoff_A (np.array): Payoff matrix for Player 1 (Rows)
    payoff_B (np.array): Payoff matrix for Player 2 (Columns)
    iterations (int): Number of steps to simulate

    Returns:
    tuple: (p1_freq_history, p2_freq_history) containing the historical
           action frequencies at each step for both players.
    """
    payoff_A = np.asarray(payoff_A, dtype=float)
    payoff_B = np.asarray(payoff_B, dtype=float)

    # Validate compatible two-player normal-form payoff matrices.
    if payoff_A.ndim != 2 or payoff_B.ndim != 2:
        raise ValueError("Both payoff matrices must be two-dimensional.")
    if payoff_A.shape != payoff_B.shape:
        raise ValueError("payoff_A and payoff_B must have the same shape.")
    if iterations < 1:
        raise ValueError("iterations must be at least 1.")

    num_actions_p1, num_actions_p2 = payoff_A.shape

    # These arrays will store the historical frequencies for plotting
    p1_freq_history = np.zeros((iterations, num_actions_p1))
    p2_freq_history = np.zeros((iterations, num_actions_p2))

    # Initialize tracking variables for the COUNT of actions taken.
    # We begin with one actual random action for each player, so the first
    # empirical belief is well-defined and never has a zero denominator.
    p1_counts = np.zeros(num_actions_p1, dtype=np.int64)
    p2_counts = np.zeros(num_actions_p2, dtype=np.int64)

    p1_initial_action = np.random.randint(num_actions_p1)
    p2_initial_action = np.random.randint(num_actions_p2)
    p1_counts[p1_initial_action] += 1
    p2_counts[p2_initial_action] += 1

    p1_freq_history[0] = p1_counts
    p2_freq_history[0] = p2_counts

    for t in range(1, iterations):
        # Historical frequencies before round t's new action.
        # There have already been exactly t observed actions for each player.
        p1_freq = p1_counts / t
        p2_freq = p2_counts / t

        # Expected payoff of every possible pure action.
        # P1 chooses a row; P2 chooses a column.
        p1_expected_payoffs = payoff_A @ p2_freq
        p2_expected_payoffs = p1_freq @ payoff_B

        # Uniformly random tie-breaking among all best responses.
        p1_best = np.flatnonzero(
            np.isclose(p1_expected_payoffs, np.max(p1_expected_payoffs),
                       rtol=1e-12, atol=1e-12)
        )
        p2_best = np.flatnonzero(
            np.isclose(p2_expected_payoffs, np.max(p2_expected_payoffs),
                       rtol=1e-12, atol=1e-12)
        )

        p1_action = np.random.choice(p1_best)
        p2_action = np.random.choice(p2_best)

        # Update action counts with the newly chosen actions.
        p1_counts[p1_action] += 1
        p2_counts[p2_action] += 1

        # Store the empirical frequencies after this round.
        # This guarantees that the final history row contains the frequencies
        # after all `iterations` actions have been counted.
        p1_freq_history[t] = p1_counts / (t + 1)
        p2_freq_history[t] = p2_counts / (t + 1)

    return p1_freq_history, p2_freq_history


# ==========================================
# PLOTTING AND EVALUATION
# ==========================================
iterations = 1_000_000

# Standard and modified payoff matrices.
A_std = np.array([
    [ 0, -1,  1],
    [ 1,  0, -1],
    [-1,  1,  0]
], dtype=float)
B_std = -A_std

A_mod = np.array([
    [ 0, -1,  3],
    [ 1,  0, -2],
    [-3,  2,  0]
], dtype=float)
B_mod = -A_mod


print("Running Standard Fictitious Play...")
np.random.seed(42)
p1_freq_history_std, p2_freq_history_std = simulate_fictitious_play(A_std, B_std, iterations)

print("Running Modified Fictitious Play...")
np.random.seed(42)
p1_freq_history_mod, p2_freq_history_mod = simulate_fictitious_play(A_mod, B_mod, iterations)

# Downsample only for drawing speed; all calculations above use the full history.
plot_step = max(1, iterations // 20_000)
plot_x = np.arange(iterations)[::plot_step]

# Set up the figure with 3 subplots
plt.figure(figsize=(18, 5))

# --- Subplot 1: Standard Game ---
plt.subplot(1, 3, 1)
plt.plot(plot_x, p1_freq_history_std[::plot_step, 0], label='P1 Cavalry')
plt.plot(plot_x, p1_freq_history_std[::plot_step, 1], label='P1 Infantry')
plt.plot(plot_x, p1_freq_history_std[::plot_step, 2], label='P1 Archers')
plt.plot(plot_x, p2_freq_history_std[::plot_step, 0], '--', alpha=0.65, label='P2 Cavalry')
plt.plot(plot_x, p2_freq_history_std[::plot_step, 1], '--', alpha=0.65, label='P2 Infantry')
plt.plot(plot_x, p2_freq_history_std[::plot_step, 2], '--', alpha=0.65, label='P2 Archers')
plt.axhline(y=1/3, color='r', linestyle='--', label='Nash (1/3)')
plt.title('Action Frequencies (Standard Game)')
plt.xlabel('Iteration')
plt.ylabel('Frequency')
plt.legend(fontsize=8)
plt.grid(True)

# --- Subplot 2: Modified Game ---
plt.subplot(1, 3, 2)
plt.plot(plot_x, p1_freq_history_mod[::plot_step, 0], label='P1 Cavalry')
plt.plot(plot_x, p1_freq_history_mod[::plot_step, 1], label='P1 Infantry')
plt.plot(plot_x, p1_freq_history_mod[::plot_step, 2], label='P1 Archers')
plt.plot(plot_x, p2_freq_history_mod[::plot_step, 0], '--', alpha=0.65, label='P2 Cavalry')
plt.plot(plot_x, p2_freq_history_mod[::plot_step, 1], '--', alpha=0.65, label='P2 Infantry')
plt.plot(plot_x, p2_freq_history_mod[::plot_step, 2], '--', alpha=0.65, label='P2 Archers')

# Theoretical Nash Equilibrium values calculated in Problem 1.2
nash_C = 1/3
nash_I = 1/2
nash_A = 1/6

plt.axhline(y=nash_C, color='r', linestyle='--', label=f'Cavalry Nash ({nash_C:.2f})')
plt.axhline(y=nash_I, color='g', linestyle='--', label=f'Infantry Nash ({nash_I:.2f})')
plt.axhline(y=nash_A, color='b', linestyle='--', label=f'Archers Nash ({nash_A:.2f})')
plt.title('Action Frequencies (Modified Game)')
plt.xlabel('Iteration')
plt.ylabel('Frequency')
plt.legend(fontsize=8)
plt.grid(True)

# --- Subplot 3: Convergence Error ---
plt.subplot(1, 3, 3)
nash_array_std = np.array([1/3, 1/3, 1/3])
nash_array_mod = np.array([nash_C, nash_I, nash_A])

dist_p1_std = np.linalg.norm(p1_freq_history_std - nash_array_std, axis=1)
dist_p2_std = np.linalg.norm(p2_freq_history_std - nash_array_std, axis=1)
dist_p1_mod = np.linalg.norm(p1_freq_history_mod - nash_array_mod, axis=1)
dist_p2_mod = np.linalg.norm(p2_freq_history_mod - nash_array_mod, axis=1)

plt.plot(plot_x, dist_p1_std[::plot_step], label='P1 Standard Error')
plt.plot(plot_x, dist_p2_std[::plot_step], '--', label='P2 Standard Error')
plt.plot(plot_x, dist_p1_mod[::plot_step], label='P1 Modified Error')
plt.plot(plot_x, dist_p2_mod[::plot_step], '--', label='P2 Modified Error')
plt.title('Convergence Distance to Nash Equilibrium')
plt.xlabel('Iteration')
plt.ylabel('Euclidean Distance')
plt.yscale('log')
plt.legend(fontsize=8)
plt.grid(True)

plt.tight_layout()
plt.show()

print("\nFinal empirical strategies:")
print("Standard P1:", p1_freq_history_std[-1],
      "distance =", dist_p1_std[-1])
print("Standard P2:", p2_freq_history_std[-1],
      "distance =", dist_p2_std[-1])
print("Modified P1:", p1_freq_history_mod[-1],
      "distance =", dist_p1_mod[-1])
print("Modified P2:", p2_freq_history_mod[-1],
      "distance =", dist_p2_mod[-1])


---
## Problem 3: Fictitious Play with Exploration (Implementation)

Our Fictitious Play agent is purely exploitative. In Reinforcement Learning, we know the importance of the **exploration-exploitation tradeoff**. Let's create an $\epsilon$-greedy version of Fictitious Play.

### 3.1 Implementation

**Your Task:** Create a new function, `simulate_epsilon_greedy_fp`. This function should be similar to your Fictitious Play implementation but include an `epsilon` parameter. At each step, with probability `epsilon`, the agent should choose a random action (explore). With probability `1-epsilon`, it should play the best response (exploit).



**Your Task:**
1.  Run the `simulate_epsilon_greedy_fp` function on the **modified** RSP game for **1,000,000 iterations** with three different `epsilon` values: `0.01`, `0.1`, and `0.3`.
2.  Plot the results for each simulation.
3.  **Analyze your results:** How does `epsilon` affect the learning dynamics? Does the agent's strategy still converge to the NE? If not, to what does it converge? Discuss the impact of exploration in this multi-agent context.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def simulate_epsilon_greedy_fp(A, B, iterations, epsilon):
    """
    Simulates epsilon-greedy Fictitious Play.
    """
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)

    if A.ndim != 2 or B.ndim != 2:
        raise ValueError("Both payoff matrices must be two-dimensional.")
    if A.shape != B.shape:
        raise ValueError("A and B must have the same shape.")
    if iterations < 1:
        raise ValueError("iterations must be at least 1.")
    if not 0.0 <= epsilon <= 1.0:
        raise ValueError("epsilon must be between 0 and 1.")

    num_actions_p1 = A.shape[0]
    num_actions_p2 = A.shape[1]

    p1_counts = np.zeros(num_actions_p1, dtype=np.int64)
    p2_counts = np.zeros(num_actions_p2, dtype=np.int64)
    p1_freq_history = np.zeros((iterations, num_actions_p1))
    p2_freq_history = np.zeros((iterations, num_actions_p2))
    p1_expected_payoff_history = np.zeros(iterations)

    # Initial random actions
    p1_action = np.random.randint(num_actions_p1)
    p2_action = np.random.randint(num_actions_p2)

    for t in range(iterations):
        # Update the action counts for both players based on their last action.
        p1_counts[p1_action] += 1
        p2_counts[p2_action] += 1

        # Calculate empirical frequencies for both players.
        p1_freq = p1_counts / (t + 1)
        p2_freq = p2_counts / (t + 1)

        # Store history (Completed for you)
        p1_freq_history[t] = p1_freq
        p2_freq_history[t] = p2_freq

        # Player 1's expected payoff under the current empirical joint strategy.
        p1_expected_payoff_history[t] = p1_freq @ A @ p2_freq

        # Expected payoff of every pure action against the opponent belief.
        p1_action_values = A @ p2_freq
        p2_action_values = p1_freq @ B

        # Epsilon-greedy logic for Player 1.
        # This exploration decision is independent of Player 2's decision.
        if np.random.rand() < epsilon:
            p1_action = np.random.randint(num_actions_p1)
        else:
            # If there are multiple best responses, break ties uniformly at random.
            p1_best = np.flatnonzero(
                np.isclose(p1_action_values, np.max(p1_action_values),
                           rtol=1e-12, atol=1e-12)
            )
            p1_action = np.random.choice(p1_best)

        # Epsilon-greedy logic for Player 2.
        if np.random.rand() < epsilon:
            p2_action = np.random.randint(num_actions_p2)
        else:
            p2_best = np.flatnonzero(
                np.isclose(p2_action_values, np.max(p2_action_values),
                           rtol=1e-12, atol=1e-12)
            )
            p2_action = np.random.choice(p2_best)

    return p1_freq_history, p2_freq_history, p1_expected_payoff_history

# Modified Warfare matrices (0: Cavalry, 1: Infantry, 2: Archers)
A_mod = np.array([
    [ 0, -1,  3],
    [ 1,  0, -2],
    [-3,  2,  0]
], dtype=float)
B_mod = -A_mod

iterations = 1_000_000
epss = [0.01, 0.1, 0.3]
nash_mod = np.array([1/3, 1/2, 1/6])
tail_window = 100_000

# Plotting block
for epsilon in epss:
    # Resetting the same seed makes the runs reproducible and comparable.
    np.random.seed(4)
    p1_freq_history, p2_freq_history, p1_payoff_history = simulate_epsilon_greedy_fp(
        A_mod, B_mod, iterations, epsilon
    )

    p1_final = p1_freq_history[-1]
    p2_final = p2_freq_history[-1]
    p1_distance = np.linalg.norm(p1_final - nash_mod)
    p2_distance = np.linalg.norm(p2_final - nash_mod)
    mean_tail_payoff = np.mean(p1_payoff_history[-tail_window:])

    # If the exploitative component were fixed at the unperturbed Nash mixture,
    # epsilon-greedy exploration would produce this simple mixture with uniform play.
    uniform = np.ones(3) / 3
    exploratory_mixture = (1 - epsilon) * nash_mod + epsilon * uniform

    print(f"\nε = {epsilon}")
    print("P1 final empirical strategy:", p1_final)
    print("P2 final empirical strategy:", p2_final)
    print("P1 distance to unperturbed Nash:", p1_distance)
    print("P2 distance to unperturbed Nash:", p2_distance)
    print(f"Mean P1 empirical-joint payoff over last {tail_window:,} rounds:", mean_tail_payoff)
    print("Fixed-component exploratory mixture (1-ε)p* + εu:", exploratory_mixture)

    # Downsample only for plotting speed.
    plot_step = max(1, iterations // 20_000)
    plot_x = np.arange(iterations)[::plot_step]

    plt.figure(figsize=(18, 5))

    plt.subplot(1, 3, 1)
    plt.plot(plot_x, p1_freq_history[::plot_step, 0], label='Cavalry')
    plt.plot(plot_x, p1_freq_history[::plot_step, 1], label='Infantry')
    plt.plot(plot_x, p1_freq_history[::plot_step, 2], label='Archers')
    plt.axhline(nash_mod[0], linestyle='--', label=f'Nash C={nash_mod[0]:.3f}')
    plt.axhline(nash_mod[1], linestyle='--', label=f'Nash I={nash_mod[1]:.3f}')
    plt.axhline(nash_mod[2], linestyle='--', label=f'Nash A={nash_mod[2]:.3f}')
    plt.title(f'P1 Frequencies (ε={epsilon})')
    plt.xlabel('Iteration')
    plt.ylabel('Frequency')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 3, 2)
    plt.plot(plot_x, p2_freq_history[::plot_step, 0], label='Cavalry')
    plt.plot(plot_x, p2_freq_history[::plot_step, 1], label='Infantry')
    plt.plot(plot_x, p2_freq_history[::plot_step, 2], label='Archers')
    plt.axhline(nash_mod[0], linestyle='--', label=f'Nash C={nash_mod[0]:.3f}')
    plt.axhline(nash_mod[1], linestyle='--', label=f'Nash I={nash_mod[1]:.3f}')
    plt.axhline(nash_mod[2], linestyle='--', label=f'Nash A={nash_mod[2]:.3f}')
    plt.title(f'P2 Frequencies (ε={epsilon})')
    plt.xlabel('Iteration')
    plt.ylabel('Frequency')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 3, 3)
    plt.plot(plot_x, p1_payoff_history[::plot_step], color='purple', label='Expected Payoff')
    plt.axhline(0, color='black', linestyle='--', linewidth=1)
    plt.title(f'P1 Expected Payoff (ε={epsilon})')
    plt.xlabel('Iteration')
    plt.ylabel('Payoff')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()


### Algorithm

Regret Matching proceeds in three main steps.

#### 1. Regret Update

Suppose Player $i$ played action $a_i$ while the opponent played action $a_{-i}$. For every possible action $s \in A_i$, update the cumulative regret as

$$
R_t(s) = R_{t-1}(s) + u_i(s, a_{-i}) - u_i(a_i, a_{-i}).
$$

where

- $R_t(s)$ is the cumulative regret for not choosing action $s$.
- $u_i(s, a_{-i})$ is the payoff that would have been received by playing $s$.
- $u_i(a_i, a_{-i})$ is the payoff actually received.

#### 2. Strategy Update

Let the positive regret be

$$
R_t^{+}(s) = \max(0, R_t(s)).
$$

The probability of selecting action $s$ in the next round is

$$
p_{t+1}(s)
=
\frac{R_t^{+}(s)}
{\sum_{s' \in A_i} R_t^{+}(s')}.
$$

If

$$
\sum_{s' \in A_i} R_t^{+}(s') = 0,
$$

then choose uniformly at random among all actions.

#### 3. Total Regret Tracking

To verify that the algorithm is working correctly, also track the **sum of Player 1's positive cumulative regrets** after each iteration.

For a no-regret algorithm, the average regret should approach zero as the number of iterations increases.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def simulate_regret_matching(A, B, iterations):
    """
    Simulates Regret Matching for two players.

    Returns:
        tuple: A tuple containing:
            - p1_avg_strat_hist (np.ndarray): History of Player 1's average strategy.
            - p1_inst_strat_hist (np.ndarray): History of Player 1's instantaneous strategy.
            - p2_avg_strat_hist (np.ndarray): History of Player 2's average strategy.
            - p2_inst_strat_hist (np.ndarray): History of Player 2's instantaneous strategy.
            - p1_total_regret_hist (np.ndarray): History of Player 1's total positive regret sum over time.
    """
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)

    if A.ndim != 2 or B.ndim != 2:
        raise ValueError("Both payoff matrices must be two-dimensional.")
    if A.shape != B.shape:
        raise ValueError("A and B must have the same shape.")
    if iterations < 1:
        raise ValueError("iterations must be at least 1.")

    num_actions_p1, num_actions_p2 = A.shape

    # Initialize histories
    p1_inst_strat_hist = np.zeros((iterations, num_actions_p1))
    p1_avg_strat_hist = np.zeros((iterations, num_actions_p1))
    p2_inst_strat_hist = np.zeros((iterations, num_actions_p2))
    p2_avg_strat_hist = np.zeros((iterations, num_actions_p2))
    p1_total_regret_hist = np.zeros(iterations)

    p1_regrets = np.zeros(num_actions_p1)
    p2_regrets = np.zeros(num_actions_p2)
    p1_strat_sum = np.zeros(num_actions_p1)
    p2_strat_sum = np.zeros(num_actions_p2)

    for t in range(iterations):
        # 1. Calculate strategies based on positive regrets.
        p1_positive_regrets = np.maximum(p1_regrets, 0.0)
        p2_positive_regrets = np.maximum(p2_regrets, 0.0)

        if p1_positive_regrets.sum() > 0:
            p1_strategy = p1_positive_regrets / p1_positive_regrets.sum()
        else:
            p1_strategy = np.ones(num_actions_p1) / num_actions_p1

        if p2_positive_regrets.sum() > 0:
            p2_strategy = p2_positive_regrets / p2_positive_regrets.sum()
        else:
            p2_strategy = np.ones(num_actions_p2) / num_actions_p2

        # 2. Store instantaneous strategies and simulate actions.
        p1_inst_strat_hist[t] = p1_strategy
        p2_inst_strat_hist[t] = p2_strategy

        p1_action = np.random.choice(num_actions_p1, p=p1_strategy)
        p2_action = np.random.choice(num_actions_p2, p=p2_strategy)

        # 3. Update cumulative regrets for EVERY counterfactual action,
        # using the opponent's action that was actually taken.
        p1_actual_payoff = A[p1_action, p2_action]
        p2_actual_payoff = B[p1_action, p2_action]

        p1_counterfactual_payoffs = A[:, p2_action]
        p2_counterfactual_payoffs = B[p1_action, :]

        p1_regrets += p1_counterfactual_payoffs - p1_actual_payoff
        p2_regrets += p2_counterfactual_payoffs - p2_actual_payoff

        # 4. Update average strategies and log total positive regret.
        p1_strat_sum += p1_strategy
        p2_strat_sum += p2_strategy

        p1_avg_strat_hist[t] = p1_strat_sum / (t + 1)
        p2_avg_strat_hist[t] = p2_strat_sum / (t + 1)

        p1_total_regret_hist[t] = np.maximum(p1_regrets, 0.0).sum()

    return p1_avg_strat_hist, p1_inst_strat_hist, p2_avg_strat_hist, p2_inst_strat_hist, p1_total_regret_hist

# Run the function using A_mod and B_mod, then plot the results.
# We also include the normalized per-round regret diagnostic requested in the assignment.
A_mod = np.array([
    [ 0, -1,  3],
    [ 1,  0, -2],
    [-3,  2,  0]
], dtype=float)
B_mod = -A_mod
nash_mod = np.array([1/3, 1/2, 1/6])

iterations = 1_000_000
np.random.seed(321)

(
    p1_avg_strat_hist,
    p1_inst_strat_hist,
    p2_avg_strat_hist,
    p2_inst_strat_hist,
    p1_total_regret_hist
) = simulate_regret_matching(A_mod, B_mod, iterations)

per_round_regret = p1_total_regret_hist / np.arange(1, iterations + 1)

print("Final P1 average strategy:", p1_avg_strat_hist[-1])
print("Final P2 average strategy:", p2_avg_strat_hist[-1])
print("P1 average-strategy distance to Nash:",
      np.linalg.norm(p1_avg_strat_hist[-1] - nash_mod))
print("P2 average-strategy distance to Nash:",
      np.linalg.norm(p2_avg_strat_hist[-1] - nash_mod))
print("Final P1 sum of positive cumulative regrets:", p1_total_regret_hist[-1])
print("Final normalized positive-regret diagnostic:",
      per_round_regret[-1])

# Downsample only for plotting speed.
plot_step = max(1, iterations // 20_000)
plot_x = np.arange(iterations)[::plot_step]

plt.figure(figsize=(18, 10))

plt.subplot(2, 2, 1)
plt.plot(plot_x, p1_inst_strat_hist[::plot_step, 0], label='Cavalry')
plt.plot(plot_x, p1_inst_strat_hist[::plot_step, 1], label='Infantry')
plt.plot(plot_x, p1_inst_strat_hist[::plot_step, 2], label='Archers')
plt.title('P1 Instantaneous Strategy')
plt.xlabel('Iteration')
plt.ylabel('Probability')
plt.legend()
plt.grid(True)

plt.subplot(2, 2, 2)
plt.plot(plot_x, p1_avg_strat_hist[::plot_step, 0], label='Cavalry')
plt.plot(plot_x, p1_avg_strat_hist[::plot_step, 1], label='Infantry')
plt.plot(plot_x, p1_avg_strat_hist[::plot_step, 2], label='Archers')
plt.axhline(nash_mod[0], linestyle='--', label=f'Nash C={nash_mod[0]:.3f}')
plt.axhline(nash_mod[1], linestyle='--', label=f'Nash I={nash_mod[1]:.3f}')
plt.axhline(nash_mod[2], linestyle='--', label=f'Nash A={nash_mod[2]:.3f}')
plt.title('P1 Running-Average Strategy')
plt.xlabel('Iteration')
plt.ylabel('Probability')
plt.legend()
plt.grid(True)

plt.subplot(2, 2, 3)
plt.plot(plot_x, p2_inst_strat_hist[::plot_step, 0], label='Cavalry')
plt.plot(plot_x, p2_inst_strat_hist[::plot_step, 1], label='Infantry')
plt.plot(plot_x, p2_inst_strat_hist[::plot_step, 2], label='Archers')
plt.title('P2 Instantaneous Strategy')
plt.xlabel('Iteration')
plt.ylabel('Probability')
plt.legend()
plt.grid(True)

plt.subplot(2, 2, 4)
plt.plot(plot_x, p2_avg_strat_hist[::plot_step, 0], label='Cavalry')
plt.plot(plot_x, p2_avg_strat_hist[::plot_step, 1], label='Infantry')
plt.plot(plot_x, p2_avg_strat_hist[::plot_step, 2], label='Archers')
plt.axhline(nash_mod[0], linestyle='--', label=f'Nash C={nash_mod[0]:.3f}')
plt.axhline(nash_mod[1], linestyle='--', label=f'Nash I={nash_mod[1]:.3f}')
plt.axhline(nash_mod[2], linestyle='--', label=f'Nash A={nash_mod[2]:.3f}')
plt.title('P2 Running-Average Strategy')
plt.xlabel('Iteration')
plt.ylabel('Probability')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(plot_x, p1_total_regret_hist[::plot_step])
plt.title('P1 Sum of Positive Cumulative Regrets')
plt.xlabel('Iteration')
plt.ylabel('Cumulative Positive Regret')
plt.grid(True)

plt.subplot(1, 2, 2)
regret_plot_x = np.arange(1, iterations + 1)[::plot_step]
plt.plot(regret_plot_x, per_round_regret[::plot_step])
plt.title('P1 Positive Regret Normalized by Time')
plt.xlabel('Iteration')
plt.ylabel(r'$\sum_s R_t^+(s) / t$')
plt.xscale('log')
plt.yscale('log')
plt.grid(True)

plt.tight_layout()
plt.show()


## Question 2: Convergence, Exploration, and Regret

### 2.1

The plots show that individual actions and instantaneous strategies may keep changing, while empirical or running-average strategies converge close to Nash.

For the modified game, fictitious play and regret matching both end very close to the equilibrium $(1/3,1/2,1/6)$. Therefore, the strongest convergence is in the **average strategy**, not in the action chosen at each round.

### 2.2

Larger $\epsilon$ causes more random actions and noisier learning. However, all tested values still finish close to the unperturbed Nash equilibrium, with mean payoff near $0$.

If exploitation settles on strategy $p$, the expected exploratory strategy is:

$$
(1-\epsilon)p+\epsilon(1/3,1/3,1/3).
$$

Thus, increasing $\epsilon$ moves the played strategy more toward uniform exploration.

### 2.3

Regret matching also shows no-regret behavior. Cumulative positive regret can increase, but the important quantity is normalized regret:

$$
\frac{\sum_s R_t^+(s)}{t}.
$$

In the simulation it falls to about $0.0047$ after $1,000,000$ iterations. Meanwhile, instantaneous strategies keep oscillating while their running averages approach Nash. This supports no-regret learning.